# Where does ff_2.1 get NuG2b's cooperativity wrong?

A companion to `nug2b-upside-analysis.ipynb`. That notebook establishes *that*
Upside under-protects NuG2b. This one asks **which** amides, and whether the
pattern has a structural explanation.

### The question has to be reframed first

"Over- vs under-predicted cooperativity" cannot be scored residue-by-residue
against Skinner, because **the experiment has no variance to fit**: all twelve
measured NH exchange through global unfolding with $m_{HX} = m_{global}$, so the
experimental cooperativity is 1.0 for every one of them. Any per-residue split of
"over" and "under" is therefore a statement about the *simulation's own* spread,
not about agreement.

So three separate questions, in order of how much they lean on experiment:

1. **Against experiment** — the sign is uniform. Confirm that, and put a number on
   the size of the deficit in the one currency that is comparable to Skinner's
   all-atom analysis.
2. **Across the sequence** — what structural descriptors predict how cooperatively
   an amide opens *in the model*? This is where the pattern lives; 32 natively
   H-bonded amides rather than 12.
3. **Relative to structural context** — after removing the trend from (2), which
   residues open *more* cooperatively than their context predicts (over) and which
   *less* (under)?

### Inherited from the parent notebook

Steps 1–3 of the parent are reproduced compactly below (load → MBAR → analysis
temperature) so this notebook runs standalone. **If the parent has already been
run in the same kernel, its objects are reused and nothing is recomputed.**

### One thing to know before reading any number

The parent's cell 10 now derives $\Delta G_{global}$ from a **van't Hoff two-state
fit** to the melting curve, replacing an earlier hard $N_{hbond}$ cutoff. The two
constructions select *different* analysis temperatures:

| construction | $T^*$ | K | below $T_m$ |
|---|---|---|---|
| van't Hoff (current code) | 0.8518 | 298.6 | 23 K |
| hard cutoff (what the parent's prose describes) | 0.7912 | 277.4 | 44 K |

The parent's step-5 and summary markdown — "~40 K below $T_m$", ESS$_{unfolded}\approx26$,
$p_{unf}\approx6\times10^{-7}$ — describes the **hard-cutoff** temperature and is
stale with respect to its own code. Everything here is computed at **both**, and
only findings that survive both are reported as findings.

In [ ]:
# Mirrors the parent's cell 0 + cell 3. nglview/ipywidgets are dropped: no
# animation here, and they make headless execution fragile.
import os
os.environ.setdefault("PYMBAR_SOLVER", "scipy")
import sys, re, glob, json
import numpy as np
import pandas as pd
import tables as tb
import mdtraj as md
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D
from scipy import stats
from scipy.special import logsumexp
from scipy.optimize import curve_fit

upside_home = os.environ["UPSIDE_HOME"]
upside_utils_dir = (os.path.join(upside_home, "py3", "py")
                    if os.path.exists(os.path.join(upside_home, "py3", "py"))
                    else os.path.join(upside_home, "py"))
if upside_utils_dir not in sys.path:
    sys.path.append(upside_utils_dir)
import mdtraj_upside as mu

import pymbar.utils, pymbar.mbar
def _lse(a, axis=None, b=None, use_numexpr=False):
    return logsumexp(a, axis=axis, b=b)
pymbar.utils.logsumexp = pymbar.mbar.logsumexp = _lse
import pymbar

base_dir  = os.path.join(os.getcwd(), os.pardir, "simulations", "upside", "nug2b") + "/"
refs_dir  = os.path.join(os.getcwd(), os.pardir, "references") + "/"
pdb_id    = "nug2b"
output_dir = base_dir + "outputs/remd_077_093_linear_ri10/"   #fme
PS_DIR     = output_dir + "protection_states/"

START_FRAME  = 500       # equilibration discard
T_REF_298    = 0.85      # ff_2.1: reduced T corresponding to 298 K
R_GAS        = 0.001987  # kcal/mol/K
UNFOLDED_CUT = 38        # N_hbond below this = unfolded
RESIDUE_FILE = PS_DIR + pdb_id + ".resid"
SKINNER_JSON = refs_dir + "skinner_2014_nug2b_hdx.json"

COOP_STRIDE = 40         #fme  frame stride for DSSP + native geometry (descriptors only)

# Skinner's own numbers for the DESRES/DESMOND trajectories, for the one
# head-to-head metric that is protocol-comparable (PNAS 2014, main text).
DESRES_F_REMAIN = 2.0 / 3.0   # "about two-thirds of the H-bonds remain intact"
DESRES_GAP      = (0.5, 2.4)  # dG_HX - dG_global for the 12 sites, kcal/mol
EXP_GAP         = (0.0, 0.8)  # same quantity, experiment

def to_kelvin(t_red):
    return t_red * 298.0 / T_REF_298

# House palette, extended from the parent. The signed-deviation figures need a
# DIVERGING scale: two hues with a neutral grey midpoint, reusing the parent's
# orange/blue pair (CVD-checked: worst adjacent dE 11.3 protan, 17.5 normal).
C_UNDER, C_MID, C_OVER = "#eb6834", "#b8b8b4", "#2a78d6"
SS_STYLE = {"H": ("#eda100", "helix"), "E": ("#4a3aa7", "strand")}
C_INK, C_GRID = "#2b2b2b", "#e3e3e0"
mpl.rcParams.update({"axes.edgecolor": "#9a9a96", "axes.labelcolor": C_INK,
                     "text.color": C_INK, "xtick.color": C_INK,
                     "ytick.color": C_INK, "axes.grid": False,
                     "figure.facecolor": "white", "savefig.facecolor": "white"})

def recessive(ax, axis="y"):
    ax.grid(True, axis=axis, color=C_GRID, lw=0.7, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

print("output_dir  %s" % os.path.relpath(output_dir, base_dir))

In [ ]:
# Load -> MBAR -> weights().  Reuses the parent's objects when this runs in the
# same kernel; otherwise rebuilds them. The rebuild is a condensed copy of the
# parent's cells 6 and 8, not a reinterpretation: same START_FRAME, same rung
# ordering (rep-major, which is the sample ordering MBAR assumes), same u_kn.
_NEED = ("E_flat", "NH_flat", "IS_OPEN", "N_CLOSED", "residues", "n_res",
         "T", "weights", "EXP", "DG_GLOBAL_EXP", "skinner", "run_files")
_INHERIT = all(k in globals() for k in _NEED)

with open(SKINNER_JSON) as fh:
    skinner = json.load(fh)
DG_GLOBAL_EXP = skinner["global"]["measurements"]["kinetics"]["delta_g"]
M_VALUE_EXP   = skinner["global"]["measurements"]["m_hx"]["value"]
first_res     = skinner["numbering"]["first_residue"]
EXP = (pd.DataFrame(skinner["residues"])
         .rename(columns={"amino_acid": "aa", "delta_g_hx": "exp", "se": "exp_se"})
         .set_index("residue").sort_index())

run_files = sorted(glob.glob(output_dir + pdb_id + ".run.[0-9]*.up"),
                   key=lambda f: int(re.search(r"run\.(\d+)\.up$", f).group(1)))

if _INHERIT:
    print("reusing the parent notebook's ensemble (%d samples)" % len(E_flat))
    HB_DONOR = globals().get("HB_DONOR")     # may not exist; rebuilt below if so
else:
    donor_resid = np.loadtxt(RESIDUE_FILE, dtype=int, ndmin=1)
    residues = pd.Index(donor_resid + first_res, name="residue")
    n_res = residues.size

    temps, energy, nhbond, protect, hbscore = [], [], [], [], []
    for i, fn in enumerate(run_files):
        with tb.open_file(fn) as t:
            temps.append(float(t.root.output.temperature[0, 0]))
            energy.append(np.asarray(t.root.output.potential[:]).ravel())
            hb = np.asarray(t.root.output.hbond[:])
            nhbond.append(hb.reshape(hb.shape[0], -1).sum(1))
            # output.hbond is (n_frame, 2*n_res): donor block then acceptor block.
            # Verified against the protection states -- within-residue corr(score,
            # protected) = +0.61 on the donor block against -/+0.27 on the acceptor
            # block, and the across-residue rank agreement is +0.73 vs -0.31.
            hbscore.append(hb[:, :n_res])
        ps_fn = next(f for f in (PS_DIR + "%s_%s_stride1_%d_PS.npy"
                                 % (pdb_id, os.path.basename(os.path.normpath(output_dir)), i),
                                 PS_DIR + "%s_%d_PS.npy" % (pdb_id, i))
                     if os.path.exists(f))
        protect.append(np.load(ps_fn))

    temps    = np.array(temps)
    n_common = min(min(len(e) for e in energy), min(p.shape[0] for p in protect))
    keep     = slice(START_FRAME, n_common)
    E_flat   = np.concatenate([e[keep] for e in energy]).astype(np.float64)
    NH_flat  = np.concatenate([h[keep] for h in nhbond]).astype(np.float64)
    _PS      = np.concatenate([p[keep] for p in protect]).astype(np.int8)
    HB_DONOR = np.concatenate([h[keep] for h in hbscore]).astype(np.float32)
    IS_OPEN  = _PS == 0
    N_CLOSED = _PS.sum(1).astype(np.float64)
    rep_of   = np.repeat(np.arange(len(run_files)), n_common - START_FRAME)
    N_k      = np.full(len(run_files), n_common - START_FRAME)
    T        = temps
    del energy, nhbond, protect, hbscore, _PS

    u_kn = np.ascontiguousarray((1.0 / T)[:, None] * E_flat[None, :])
    mbar = pymbar.MBAR(u_kn, N_k)
    _log_denom = logsumexp(mbar.f_k[:, None] - u_kn,
                           b=N_k[:, None].astype(float), axis=0)

    def weights(t_red):
        """Normalised weight of every pooled frame at t_red."""
        lw = -(E_flat / t_red) - _log_denom
        w = np.exp(lw - lw.max())
        return w / w.sum()

    print("%d rungs, T %.3f..%.3f (%.0f..%.0f K), %d pooled samples, %d NH donors"
          % (len(T), T.min(), T.max(), to_kelvin(T.min()), to_kelvin(T.max()),
             len(E_flat), n_res))

# The donor-side H-bond score is a descriptor this notebook adds; the parent does
# not keep it, so read it if the inherited path skipped it.
if HB_DONOR is None:
    _sc = []
    for fn in run_files:
        with tb.open_file(fn) as t:
            _sc.append(np.asarray(t.root.output.hbond[:])[:, :n_res])
    _n = len(E_flat) // len(run_files) + START_FRAME
    HB_DONOR = np.concatenate([s[START_FRAME:_n] for s in _sc]).astype(np.float32)
    del _sc
if "rep_of" not in globals():
    rep_of = np.repeat(np.arange(len(run_files)), len(E_flat) // len(run_files))
assert HB_DONOR.shape == (len(E_flat), n_res), HB_DONOR.shape

In [ ]:
# Both analysis-temperature constructions, so every result below can be checked
# against the one the parent's prose assumes as well as the one its code uses.
def two_state(TK, aN, bN, YU, dH, dS):
    K = np.exp(np.clip(-(dH - TK * dS) / (R_GAS * TK), -500.0, 500.0))
    fU = K / (1.0 + K)
    return (aN + bN * TK) * (1.0 - fU) + YU * fU

TK_rungs = to_kelvin(T)
W_T  = [weights(t) for t in T]
_p, _ = curve_fit(two_state, TK_rungs, np.array([float(w @ NH_flat) for w in W_T]),
                  p0=(76.0, -0.07, 12.0, 110.0, 0.35), maxfev=200000)
dH_vh, dS_vh = _p[3], _p[4]
T_MELT = float(dH_vh / dS_vh * T_REF_298 / 298)
T_VH   = float((dH_vh - DG_GLOBAL_EXP) / dS_vh * T_REF_298 / 298)

def global_stability_cut(t_red, w=None):
    """dG_global from the hard N_hbond cut -- the parent's earlier construction."""
    w = weights(t_red) if w is None else w
    p_unf = w[NH_flat < UNFOLDED_CUT].sum()
    if not 0.0 < p_unf < 1.0:
        return np.nan
    return R_GAS * to_kelvin(t_red) * np.log((1.0 - p_unf) / p_unf)

_ts = np.arange(T.min(), T.max() + 1e-9, 0.002)
_dg = np.array([global_stability_cut(t) for t in _ts])
_ok = np.isfinite(_dg)
T_CUT = float(np.interp(-DG_GLOBAL_EXP, -_dg[_ok], _ts[_ok]))

T_SET = {"van't Hoff": T_VH, "hard cut": T_CUT}
print("T_m         %.4f  (%.1f K)" % (T_MELT, to_kelvin(T_MELT)))
for k, v in T_SET.items():
    print("%-11s %.4f  (%.1f K)   %2.0f K below T_m   dG_cut there = %.2f"
          % (k, v, to_kelvin(v), to_kelvin(T_MELT) - to_kelvin(v),
             global_stability_cut(v)))
print("\nthe parent's step-5/summary prose is written for the hard-cut T; its code"
      "\nselects the van't Hoff T. Both are carried through below.")

## Three ways to measure cooperativity, and what each is comparable to

For amide $j$, let $N_{closed}$ be the number of protected NH in a frame and let
the ensemble average be taken under the MBAR weights at the analysis temperature.

**1. H-bonds broken on opening.**

$$\Delta N_{HB}(j) \;=\; \langle N_{closed}\rangle_{j\ \rm closed} \;-\; \langle N_{closed}\rangle_{j\ \rm open}$$

This is Peng's $m_{total}/k$ *identically* — his $m_{total} = m_{closed} - m_{open}
= k\,[(N_c - S) - (N_o - S)] = k(N_c - N_o)$ — so the denaturant slope $s$ cancels
exactly at $[den]=0$ and this quantity is **immune to the parent's failed $s$
calibration**. It is the honest form of that measurement: a conditional H-bond
count, no denaturant model in it at all.

**2. Cooperativity index** $\varphi(j) = \Delta N_{HB}(j) / \Delta N_{global}$,
where $\Delta N_{global}$ is the same difference between the folded and unfolded
sub-ensembles. $\varphi = 1$ means opening $j$ costs exactly what global unfolding
costs; $\varphi \approx 0$ means a purely local fluctuation. Experiment puts all
twelve measured NH at $\varphi = 1$. Note $\Delta N_{global}$ *is* fit-dependent
in the parent's route classification (it is read at 1 M, outside linear response
once $s$ is inflated); here it is measured directly at $[den] = 0$, so $\varphi$
is fit-free too.

**3. Skinner's surviving-H-bond fraction**, the one number in the paper that can
be compared across the two force fields without sharing a protocol:

$$f_{remain}(j) = \frac{\langle N_{closed}\rangle_{j\ \rm open}}{\langle N_{closed}\rangle_{\rm folded} - 1}$$

Skinner reports $f_{remain} \approx 2/3$ for the DESRES trajectories and
$\approx 0$ for experiment. **The conditioning matters and is easy to get wrong.**
Skinner measures it *inside the DSE* ("when any one of the 12 H-bonds is broken
for 1+ ns in DSE$_{sim}$"), which is a statement about residual structure in the
denatured state. Conditioning on "$j$ open" over the whole ensemble instead
measures something different — how local the *dominant* opening channel is under
native conditions. Both are computed below and kept apart:

- `f_remain` — whole ensemble. Answers "is the channel that actually carries the
  exchange flux local or global?"
- `f_remain_dse` — restricted to $N_{hbond} <$ `UNFOLDED_CUT`. **This is the one
  that compares to Skinner's 2/3.**

In [ ]:
def cooperativity(t_red):
    """Per-residue cooperativity at t_red. Returns a frame indexed by residue
    label, with the ensemble-wide references in .attrs."""
    w   = weights(t_red)
    RT  = R_GAS * to_kelvin(t_red)
    unf = NH_flat < UNFOLDED_CUT
    wf, wu = w[~unf], w[unf]

    # References: the folded and unfolded sub-ensembles, same currency as dN_HB.
    N_fold = float((wf * N_CLOSED[~unf]).sum() / wf.sum())
    N_unf  = float((wu * N_CLOSED[unf]).sum() / wu.sum())
    dN_global = N_fold - N_unf
    dG_cut    = RT * np.log((1.0 - wu.sum()) / wu.sum())

    S_all = float((w * N_CLOSED).sum())
    rec = {}
    for j in range(n_res):
        m  = IS_OPEN[:, j]
        wj = w[m]
        p_o = wj.sum()
        S_o = float((wj * N_CLOSED[m]).sum())
        # <N_closed | j open> and | j closed>. j itself is excluded from the first
        # by construction (it is open there) and subtracted from the second.
        N_o = S_o / p_o if p_o > 0 else np.nan
        N_c = (S_all - S_o) / (1.0 - p_o) if p_o < 1 else np.nan

        md_ = m & unf                          # j open AND the chain unfolded
        wd  = w[md_]
        N_o_dse = float((wd * N_CLOSED[md_]).sum() / wd.sum()) if wd.sum() > 0 else np.nan
        ess_dse = (wd.sum() ** 2 / np.sum(wd ** 2)) if np.sum(wd ** 2) > 0 else 0.0

        with np.errstate(divide="ignore", invalid="ignore"):
            dg = RT * np.log1p(-p_o) - RT * np.log(p_o) if 0 < p_o < 1 else np.nan
        rec[residues[j]] = dict(
            p_open=p_o, n_open_raw=int(m.sum()),
            ess_open=(p_o ** 2 / np.sum(wj ** 2)) if wj.size and np.sum(wj ** 2) > 0 else 0.0,
            dg=dg, N_open=N_o, N_closed_cond=N_c,
            dN_HB=N_c - N_o,
            f_remain=N_o / (N_fold - 1.0),
            f_remain_dse=N_o_dse / (N_fold - 1.0),
            ess_open_dse=ess_dse)

    A = pd.DataFrame(rec).T
    A.index.name = "residue"
    A["phi"]      = A["dN_HB"] / dN_global
    A["resolved"] = (A["n_open_raw"] >= 5) & (A["ess_open"] >= 5)
    A["gap"]      = A["dg"] - dG_cut          # the Skinner Fig. 3 currency
    A.attrs.update(T=t_red, RT=RT, N_fold=N_fold, N_unf=N_unf,
                   dN_global=dN_global, dG_cut=dG_cut, p_unf=float(wu.sum()),
                   ess=float(1.0 / np.sum(w ** 2)))
    return A

COOP = {k: cooperativity(v) for k, v in T_SET.items()}
for k, A in COOP.items():
    print("%-11s T=%.4f  N_fold %.1f  N_unf %.1f  dN_global %.1f  dG_cut %.2f  "
          "p_unf %.2e  ESS %.0f"
          % (k, A.attrs["T"], A.attrs["N_fold"], A.attrs["N_unf"],
             A.attrs["dN_global"], A.attrs["dG_cut"], A.attrs["p_unf"],
             A.attrs["ess"]))

## Structural descriptors

Six candidate explanations, all computed from the *folded* frames so they describe
the native structure rather than the thing being explained:

| descriptor | hypothesis it tests |
|---|---|
| `hb_score_nat` | a weakly H-bonded amide frays first (native H-bond strength) |
| `sep` | non-local H-bonds need global unfolding; $i,i{+}4$ helix bonds do not |
| `nbr10` | buried amides open only globally (burial) |
| `elem_edge` | secondary-structure elements unzip from their ends |
| `term_dist` | the chain frays from its termini |
| `ss` / `elem` | helix and sheet fail differently |

`sep` comes from the modal nearest backbone O to each NH over folded frames — the
Upside topology stores donor and acceptor lists but not the pairing, so it is
measured rather than looked up. Amides with no partner within 3.5 Å in at least
20% of folded frames get `sep = NaN`; those are the outward-facing amides of the
sheet, which are never protected and are excluded from the analysis anyway.

In [ ]:
traj = globals().get("traj")
if traj is None:
    traj = mu.load_upside_rep(run_files, 0)
sub  = traj[::COOP_STRIDE]
dssp = md.compute_dssp(sub, simplified=True)

# Folded frames, judged the same way the parent's cell 14 judges them. Note this
# reuses UNFOLDED_CUT against a COUNT OF STRUCTURED RESIDUES, whereas the MBAR
# path applies it to the summed H-bond score -- two different quantities that the
# parent happens to threshold at the same number. Kept for consistency with it.
folded_f = ((dssp == "H") | (dssp == "E")).sum(1) >= UNFOLDED_CUT
ss_frames = dssp[folded_f] if folded_f.any() else dssp
ss = pd.Series(np.array(list("HEC"))[
                   np.stack([(ss_frames == c).sum(0) for c in "HEC"]).argmax(0)],
               index=np.arange(dssp.shape[1]) + first_res, name="ss")
fold = sub[folded_f] if folded_f.any() else sub
print("DSSP on %d frames (stride %d): %d folded (%.0f%%)"
      % (len(sub), COOP_STRIDE, folded_f.sum(), 100 * folded_f.mean()))

top   = fold.top
nh_at = {top.atom(i).residue.index: i for i in top.select("name NH")}
o_at  = {top.atom(i).residue.index: i for i in top.select("name O")}
sc_at = {top.atom(i).residue.index: i for i in top.select("name CB or name CA")}

# Native H-bond partner: modal nearest acceptor O to each NH, |i-j| > 1, < 3.5 A.
don_ri, acc_ri = np.array(sorted(nh_at)), np.array(sorted(o_at))
_pairs = np.array([[nh_at[d], o_at[a]] for d in don_ri for a in acc_ri])
D = md.compute_distances(fold, _pairs).reshape(len(fold), len(don_ri), len(acc_ri)) * 10.0
partner, partner_frac = {}, {}
for k, d in enumerate(don_ri):
    dd = D[:, k, :].copy()
    dd[:, np.abs(acc_ri - d) <= 1] = np.inf          # self / adjacent is not an H-bond
    near = np.argmin(dd, axis=1)
    ok   = dd[np.arange(len(dd)), near] < 3.5
    r    = d + first_res
    if ok.sum() < 0.2 * len(dd):
        partner[r], partner_frac[r] = np.nan, 0.0
        continue
    v, c = np.unique(near[ok], return_counts=True)
    partner[r]      = acc_ri[v[np.argmax(c)]] + first_res
    partner_frac[r] = c.max() / len(dd)
del D

# Burial proxy: sidechain-centroid neighbours within 10 A, folded mean.
sc_ri  = np.array(sorted(sc_at))
_scp   = np.array([[sc_at[a], sc_at[b]]
                   for i, a in enumerate(sc_ri) for b in sc_ri[i + 1:]])
_scd   = md.compute_distances(fold, _scp).mean(0) * 10.0
nbr10  = pd.Series(0.0, index=sc_ri + first_res)
for (a, b), dist in zip([(a, b) for i, a in enumerate(sc_ri) for b in sc_ri[i + 1:]], _scd):
    if dist < 10.0:
        nbr10[a + first_res] += 1
        nbr10[b + first_res] += 1

DESC = pd.DataFrame(index=residues)
DESC["ss"]           = ss.reindex(DESC.index)
DESC["partner"]      = pd.Series(partner).reindex(DESC.index)
DESC["partner_frac"] = pd.Series(partner_frac).reindex(DESC.index)
DESC["sep"]          = (DESC["partner"] - DESC.index.to_series()).abs()
DESC["nbr10"]        = nbr10.reindex(DESC.index)
DESC["term_dist"]    = np.minimum(DESC.index.to_series() - residues.min(),
                                  residues.max() - DESC.index.to_series())

# Element label and distance to the nearer end of that element.
_seg = ((DESC["ss"] != DESC["ss"].shift())
        | (DESC.index.to_series().diff() != 1)).cumsum()
_elem, _edge, _lab = {}, {}, 0
for _, grp in DESC.groupby(_seg):
    if grp["ss"].iloc[0] == "C":
        for r in grp.index:
            _elem[r], _edge[r] = "coil", 0
        continue
    _lab += 1
    nm = "%s%d" % (grp["ss"].iloc[0], _lab)
    for pos, r in enumerate(grp.index):
        _elem[r], _edge[r] = nm, min(pos, len(grp) - 1 - pos)
DESC["elem"]      = pd.Series(_elem)
DESC["elem_edge"] = pd.Series(_edge)

# Native H-bond score, from the coldest rung with no reweighting.
_cold = int(np.argmin(T))
DESC["hb_score_nat"] = pd.Series(HB_DONOR[rep_of == _cold].mean(0), index=residues)

# The analysis set: amides that are actually H-bonded in the native state. The
# rest are the outward-facing sheet amides -- never protected, so "how
# cooperatively do they open" is not a question about them.
DESC["native_protected"] = DESC["hb_score_nat"] > 0.5
print("native-protected amides: %d of %d   (all %d measured NH included: %s)"
      % (DESC["native_protected"].sum(), len(DESC), len(EXP),
         bool(DESC.loc[EXP.index, "native_protected"].all())))

# join() does not carry .attrs across -- the same trap the parent works around in
# its cell 12 -- so the ensemble references are detached and reattached.
for k in COOP:
    _at = dict(COOP[k].attrs)
    COOP[k] = COOP[k].join(DESC).join(EXP)
    COOP[k].attrs.update(_at)
    COOP[k]["err"] = COOP[k]["dg"] - COOP[k]["exp"]
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 90)
DESC.round(3)

## Q1 — against experiment the sign is uniform

Every measured NH comes out less cooperative than experiment; there is no
"over-predicted" residue at the experimental anchor. What varies is *how much*,
and the two Skinner-comparable numbers show the deficit has two distinct parts.

In [ ]:
for k, A in COOP.items():
    M = A[A["exp"].notna()]
    R = M[M["resolved"]]
    print("=" * 96)
    print("%s   T=%.4f (%.1f K)   dN_global=%.1f" %
          (k, A.attrs["T"], to_kelvin(A.attrs["T"]), A.attrs["dN_global"]))
    print(M[["aa", "ss", "elem", "elem_edge", "nbr10", "sep", "hb_score_nat",
             "dg", "exp", "err", "gap", "dN_HB", "phi", "f_remain",
             "f_remain_dse", "ess_open", "ess_open_dse", "resolved"]]
          .round(3).to_string())
    print("resolved %d/%d   dG_HX RMSE %.2f  bias %.2f   mean phi %.2f"
          % (len(R), len(M), np.sqrt((R["err"] ** 2).mean()), R["err"].mean(),
             R["phi"].mean()))
    print("mean f_remain (whole ensemble) %.3f   mean f_remain_dse (Skinner-comparable) %.3f"
          % (R["f_remain"].mean(), R["f_remain_dse"].mean()))
    print("phi > 1: %s      dN_HB < 0: %s"
          % (list(A.index[A["phi"] > 1]), list(A.index[A["dN_HB"] < 0])))

In [ ]:
# Skinner's Fig. 3 currency (dG_HX referenced to dG_global) and his surviving-
# H-bond fraction, both against the DESRES numbers. One measure per panel, one
# axis each -- the two quantities are not put on a shared scale.
A = COOP["van't Hoff"]
M = A[A["exp"].notna() & A["resolved"]].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))

ax = axes[0]
recessive(ax)
ax.axhspan(*EXP_GAP, color=C_MID, alpha=0.45, lw=0, zorder=1)
ax.axhspan(*DESRES_GAP, color=C_OVER, alpha=0.16, lw=0, zorder=1)
ax.axhline(0, color="#9a9a96", lw=0.9, zorder=2)
ax.bar(np.arange(len(M)), M["gap"], color=C_UNDER, width=0.7, zorder=3)
ax.set_xticks(np.arange(len(M)))
ax.set_xticklabels(["%d%s" % (r, M.at[r, "aa"]) for r in M.index], fontsize=8)
# Headroom above the DESMOND band so the legend never sits on a bar.
ax.set_ylim(min(M["gap"].min() * 1.18, -0.4), 5.2)
ax.set_ylabel(r"$\Delta G_{HX} - \Delta G_{global}$  (kcal/mol)")
ax.set_xlabel("measured amide")
ax.set_title("Skinner Fig. 3 currency: Upside falls below the global line", fontsize=10)
ax.legend(handles=[Patch(color=C_MID, alpha=0.45, label="experiment  0 to +0.8"),
                   Patch(color=C_OVER, alpha=0.16, label="DESMOND  +0.5 to +2.4"),
                   Patch(color=C_UNDER, label="Upside ff_2.1")],
          frameon=False, fontsize=8, loc="upper left", ncol=1)

ax = axes[1]
recessive(ax)
ax.axhline(DESRES_F_REMAIN, color=C_OVER, lw=2, ls="--", zorder=2)
ax.axhline(0.0, color="#6b6b67", lw=1.4, zorder=2)
ax.bar(np.arange(len(M)) - 0.19, M["f_remain"], width=0.36,
       color=C_UNDER, label="Upside, whole ensemble", zorder=3)
ax.bar(np.arange(len(M)) + 0.19, M["f_remain_dse"], width=0.36,
       color=C_MID, label="Upside, restricted to the DSE", zorder=3)
ax.set_xticks(np.arange(len(M)))
ax.set_xticklabels(["%d%s" % (r, M.at[r, "aa"]) for r in M.index], fontsize=8)
ax.set_ylim(-0.06, 1.52)
ax.set_ylabel(r"$f_{remain}$: fraction of other H-bonds intact")
ax.set_xlabel("measured amide")
ax.set_title("two conditionings, two different failures", fontsize=10)
# The two reference levels go in the legend rather than as in-plot text, which
# would land on the bars at every x position.
ax.legend(handles=[Patch(color=C_UNDER, label="Upside, whole ensemble"),
                   Patch(color=C_MID, label="Upside, restricted to the DSE"),
                   Line2D([], [], color=C_OVER, lw=2, ls="--",
                          label="DESMOND, in its DSE (0.67)"),
                   Line2D([], [], color="#6b6b67", lw=1.4,
                          label=r"experiment ($\approx$ 0)")],
          frameon=False, fontsize=8, loc="upper left", ncol=2)

plt.tight_layout(); plt.show()

print("mean f_remain  whole ensemble %.3f  |  DSE-restricted %.3f  |  DESMOND %.2f  |  exp ~0"
      % (M["f_remain"].mean(), M["f_remain_dse"].mean(), DESRES_F_REMAIN))

### What the two panels say

- **Left** — all twelve sit at or below the global line, against experiment's
  $0$ to $+0.8$ and DESMOND's $+0.5$ to $+2.4$. Upside's error is the opposite
  sign to Shaw's and larger in magnitude.
- **Right** — the two conditionings separate two failures that are easy to conflate:
  - `f_remain_dse` $\approx 0.80$ against DESMOND's $0.67$ and experiment's $\approx 0$.
    On **residual structure in the denatured state**, Upside fails the *same way*
    as the all-atom trajectories Skinner criticised, and somewhat worse.
  - `f_remain` $\approx 0.96$ over the whole ensemble. This is the failure DESMOND
    does *not* have: under native conditions the flux is carried by a **local
    fray in an otherwise intact native state**, not by the DSE at all. This is
    what drives $\Delta G_{HX}$ ~4.3 kcal/mol below experiment.

## Q2 — what predicts cooperativity across the sequence

32 natively H-bonded amides instead of 12, and a graded response instead of a
constant one. Reported at both analysis temperatures; a descriptor that only
works at one of them is not a finding.

In [ ]:
CANDIDATES = ["hb_score_nat", "sep", "nbr10", "elem_edge", "term_dist",
              "partner_frac"]
rows = []
for k, A in COOP.items():
    g = A[A["native_protected"] & A["resolved"]]
    for c in CANDIDATES + ["dg", "p_open"]:
        x = g[c].astype(float)
        ok = x.notna() & g["dN_HB"].notna()
        if ok.sum() < 6:
            continue
        rs, ps = stats.spearmanr(x[ok], g.loc[ok, "dN_HB"])
        rows.append(dict(T=k, descriptor=c, n=int(ok.sum()),
                         spearman=rs, p=ps))
CORR = pd.DataFrame(rows).pivot(index="descriptor", columns="T",
                                values=["spearman", "p", "n"])
CORR = CORR.loc[CANDIDATES + ["dg", "p_open"]]
print("Spearman correlation with dN_HB, native-protected & resolved amides\n")
print(CORR.round(3).to_string())

# Which survive both temperatures at p < 0.05?
_sp = CORR["p"]
robust = [d for d in CANDIDATES if (_sp.loc[d] < 0.05).all()]
print("\nsignificant at p<0.05 under BOTH constructions: %s" % (robust or "none"))
print("significant under the van't Hoff T only: %s"
      % [d for d in CANDIDATES
         if _sp.loc[d, "van't Hoff"] < 0.05 and _sp.loc[d, "hard cut"] >= 0.05])
print("\n%d descriptors tested x 2 temperatures: at p<0.05 one false positive is"
      "\nexpected by chance, so single-temperature hits are leads, not results."
      % len(CANDIDATES))

In [ ]:
# hb_score_nat and p_open are not independent -- a weakly H-bonded amide opens
# more often -- so check whether hb_score_nat still explains dN_HB once the
# opening probability is held fixed. Spearman partial correlation via residuals
# of the rank variables.
def partial_spearman(x, y, z):
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz)
    ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    return stats.pearsonr(ex, ey)

for k, A in COOP.items():
    g = A[A["native_protected"] & A["resolved"]].dropna(
        subset=["hb_score_nat", "dN_HB", "p_open"])
    r0, p0 = stats.spearmanr(g["hb_score_nat"], g["dN_HB"])
    r1, p1 = partial_spearman(g["hb_score_nat"], g["dN_HB"], np.log(g["p_open"]))
    print("%-11s n=%2d   hb_score_nat vs dN_HB: rho %+.3f (p=%.4f)   "
          "controlling for log p_open: %+.3f (p=%.4f)"
          % (k, len(g), r0, p0, r1, p1))
print("\nIf the partial correlation collapses, hb_score_nat is not an independent"
      "\nexplanation -- it acts through how often the amide opens at all.")

In [ ]:
# Grouped view: element by element, and the position-within-element gradient.
for k, A in COOP.items():
    prot = A[A["native_protected"]]
    print("=" * 78)
    print("%s   (dN_global = %.1f)" % (k, A.attrs["dN_global"]))
    print(prot.groupby("elem")[["dN_HB", "phi", "f_remain_dse", "dg", "gap"]]
          .agg(["size", "mean"]).round(3).to_string())
    print("\nby distance to the nearer end of its own element:")
    print(prot.groupby("elem_edge")[["dN_HB", "phi"]]
          .agg(["size", "mean"]).round(3).to_string())

In [ ]:
# Panel 1: phi along the sequence, with the native-structure track the parent uses.
# One series, so no legend box -- the title names it and the reference line is
# labelled in place. Panel 2 and 3: the two descriptors that survived, coloured by
# secondary structure (identity is also carried by the direct labels).
A  = COOP["van't Hoff"]
pr = A[A["native_protected"]]
# constrained_layout, not tight_layout: the SS track uses add_patch in data
# coordinates, which tight_layout cannot measure.
fig = plt.figure(figsize=(13.5, 4.1), constrained_layout=True)
gs  = fig.add_gridspec(1, 3, width_ratios=[1.55, 1, 1])

ax = fig.add_subplot(gs[0, 0])
recessive(ax)
ax.axhline(1.0, color=C_OVER, lw=1.6, ls="--", zorder=2)
ax.bar(pr.index, pr["phi"], width=0.8, color=C_UNDER, zorder=3)
_m = pr[pr["exp"].notna()]
ax.plot(_m.index, _m["phi"], "k*", ms=9, ls="none", zorder=4)
lo   = min(0.0, pr["phi"].min())
band = 0.05 * (pr["phi"].max() - lo)
by   = lo - 1.8 * band
# Native-structure track, labelled with the SAME element names the tables use so
# the two can be read against each other.
for nm, grp in A[A["elem"] != "coil"].groupby("elem"):
    x0, x1 = grp.index.min() - 0.5, grp.index.max() + 0.5
    ax.add_patch(Rectangle((x0 + 0.07, by), (x1 - x0) - 0.14, band, lw=0,
                           color=SS_STYLE[grp["ss"].iloc[0]][0], zorder=3))
    ax.annotate(nm, ((x0 + x1) / 2, by), xytext=(0, -11),
                textcoords="offset points", ha="center", fontsize=7.5,
                color="#6b6b67")
ax.set_ylim(by - 3.4 * band, pr["phi"].max() * 1.30)
# Below the line, over the low-phi N-terminal bars: above it would cross H3.
ax.annotate("global unfolding = experiment",
            (pr.index.min() - 0.5, 1.0), xytext=(0, -13),
            textcoords="offset points", ha="left", fontsize=8, color=C_OVER)
ax.set_xlabel("residue"); ax.set_ylabel(r"$\varphi = \Delta N_{HB}/\Delta N_{global}$")
ax.set_title("cooperativity along the sequence (stars = measured)", fontsize=10)

for gsi, (xc, xlab) in enumerate([("elem_edge", "distance to nearer end of its element"),
                                  ("hb_score_nat", "native H-bond score")]):
    ax = fig.add_subplot(gs[0, gsi + 1])
    recessive(ax, axis="both")
    ax.axhline(1.0, color=C_OVER, lw=1.2, ls="--", zorder=2)
    for s, (col, lbl) in SS_STYLE.items():
        q = pr[pr["ss"] == s]
        ax.plot(q[xc], q["phi"], "o", ms=7, mfc=col, mec="white", mew=1.1,
                ls="none", label=lbl, zorder=3)
    q = pr[~pr["ss"].isin(SS_STYLE)]
    ax.plot(q[xc], q["phi"], "o", ms=7, mfc=C_MID, mec="white", mew=1.1,
            ls="none", label="coil", zorder=3)
    for r in pr.index[pr["exp"].notna()]:
        ax.annotate("%d" % r, (pr.at[r, xc], pr.at[r, "phi"]), fontsize=7,
                    xytext=(4, 3), textcoords="offset points", color="#4a4a46")
    rho, pv = stats.spearmanr(pr[xc].astype(float), pr["phi"])
    ax.set_xlabel(xlab); ax.set_ylabel(r"$\varphi$")
    ax.set_ylim(pr["phi"].min() - 0.12, pr["phi"].max() * 1.22)
    ax.set_title(r"$\rho$ = %+.2f  (p = %.3f)" % (rho, pv), fontsize=10)
    ax.legend(frameon=False, fontsize=8, loc="upper left")

plt.show()

## Q3 — over vs under, relative to structural context

With the experimental target flat at $\varphi = 1$, the only well-posed
per-residue "over vs under" is against the model's own structural trend: fit
$\Delta N_{HB}$ on the descriptors that survived Q2, then read the signed
residual. Positive = opens *more* cooperatively than its structural context
predicts; negative = frays *more* than its context predicts.

This is a within-model diagnostic. It localises where the force field departs
from its own regularity, which is where a fix would have to act — it is not a
comparison to experiment.

In [ ]:
# Ordinary least squares on the surviving descriptors, via lstsq (no statsmodels
# dependency). Deliberately small: 32 points will fit anything given enough terms.
FEATURES = ["hb_score_nat", "elem_edge"]

def fit_trend(A):
    g = A[A["native_protected"] & A["resolved"]].dropna(subset=FEATURES + ["dN_HB"])
    X = np.column_stack([np.ones(len(g))] + [g[f].astype(float) for f in FEATURES])
    y = g["dN_HB"].to_numpy(float)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ coef
    ss_res, ss_tot = ((y - pred) ** 2).sum(), ((y - y.mean()) ** 2).sum()
    return g.index, coef, pred, y - pred, 1 - ss_res / ss_tot

for k, A in COOP.items():
    idx, coef, pred, resid, r2 = fit_trend(A)
    A.loc[idx, "trend"] = pred
    A.loc[idx, "resid"] = resid
    print("%-11s n=%d  R2=%.2f   dN_HB ~ %.2f %+.2f*hb_score_nat %+.2f*elem_edge"
          % (k, len(idx), r2, coef[0], coef[1], coef[2]))

A = COOP["van't Hoff"]
R = A.dropna(subset=["resid"]).sort_values("resid")
print("\nmost cooperative for their context (top 5):")
print(R.tail(5)[["ss", "elem", "elem_edge", "hb_score_nat", "dN_HB", "trend",
                 "resid", "phi", "exp"]].round(2).to_string())
print("\nmost frayed for their context (bottom 5):")
print(R.head(5)[["ss", "elem", "elem_edge", "hb_score_nat", "dN_HB", "trend",
                 "resid", "phi", "exp"]].round(2).to_string())

In [ ]:
# Diverging figure: signed residual, two hues with a neutral grey midpoint.
# Sign is also carried by which side of the baseline the bar sits on, so colour
# is not the only encoding.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.9),
                         gridspec_kw={"width_ratios": [1.5, 1]})

for k, ax in zip(COOP, axes):
    A = COOP[k]
    R = A.dropna(subset=["resid"])
    recessive(ax)
    ax.axhline(0, color="#9a9a96", lw=1.0, zorder=2)
    ax.bar(R.index, R["resid"], width=0.8, zorder=3,
           color=[C_OVER if v > 0 else C_UNDER for v in R["resid"]])
    # Label the 7 largest deviations, staggering the offset when two labelled
    # residues are sequence neighbours so the text does not overlap.
    big  = sorted(R["resid"].abs().sort_values().index[-7:])
    prev = None
    for n, r in enumerate(big):
        v    = R.at[r, "resid"]
        near = prev is not None and r - prev <= 2
        off  = (18 if near else 5) if v > 0 else (-22 if near else -12)
        ax.annotate("%d%s" % (r, R.at[r, "ss"]), (r, v), ha="center",
                    xytext=(0, off), textcoords="offset points",
                    fontsize=7.5, color="#4a4a46")
        prev = r
    mk = R.index[R["exp"].notna()]
    ax.plot(mk, R.loc[mk, "resid"], "k*", ms=8, ls="none", zorder=4)
    pad = 0.42 * max(abs(R["resid"].min()), abs(R["resid"].max()))
    ax.set_ylim(R["resid"].min() - pad, R["resid"].max() + pad)
    ax.set_xlabel("residue")
    ax.set_ylabel(r"$\Delta N_{HB}$ - structural trend")
    ax.set_title("%s T   (stars = measured)" % k, fontsize=10)
    if ax is axes[0]:
        ax.legend(handles=[Patch(color=C_OVER, label="more cooperative than context"),
                           Patch(color=C_UNDER, label="frays more than context")],
                  frameon=False, fontsize=8, loc="upper left")

plt.tight_layout(); plt.show()

# Do the two temperatures agree on the ranking? That is the test of whether this
# localisation is a property of the force field or of the analysis temperature.
_a = COOP["van't Hoff"]["resid"].dropna()
_b = COOP["hard cut"]["resid"].dropna()
_i = _a.index.intersection(_b.index)
print("residual ranking agreement between the two analysis temperatures: "
      "Spearman %+.3f over %d amides"
      % (stats.spearmanr(_a[_i], _b[_i]).statistic, len(_i)))

In [ ]:
# How much of this rests on how many frames? dN_HB for the extreme residues, with
# a moving-block bootstrap over contiguous frame blocks.
#
# CAVEAT: MBAR's f_k are held FIXED across resamples, and blocks are contiguous
# chunks of the rep-major pooled array (so a block lies within one rung). This
# captures within-rung correlation but not the rung-to-rung coupling MBAR
# introduces, so these intervals are a LOWER BOUND on the true uncertainty.
A = COOP["van't Hoff"]
w = weights(T_SET["van't Hoff"])
FOCUS = [3, 4, 15, 51, 53, 55, 5, 7, 16, 26, 29, 30, 31, 34]
NBLK, NBOOT = 40, 200
rng = np.random.default_rng(0)
blk = np.minimum(np.arange(len(w)) // (len(w) // NBLK), NBLK - 1)
blk_idx = [np.flatnonzero(blk == q) for q in range(NBLK)]
cols = {r: list(residues).index(r) for r in FOCUS}

boot = {r: [] for r in FOCUS}
for _ in range(NBOOT):
    idx = np.concatenate([blk_idx[q] for q in rng.integers(0, NBLK, NBLK)])
    wb  = w[idx]; wb /= wb.sum()
    Nb  = N_CLOSED[idx]
    S   = float((wb * Nb).sum())
    for r, j in cols.items():
        m  = IS_OPEN[idx, j]
        wj = wb[m]
        p_o = wj.sum()
        if not 0 < p_o < 1:
            boot[r].append(np.nan); continue
        S_o = float((wj * Nb[m]).sum())
        boot[r].append((S - S_o) / (1 - p_o) - S_o / p_o)

BOOT = pd.DataFrame({
    r: dict(dN_HB=A.at[r, "dN_HB"],
            lo=np.nanpercentile(boot[r], 2.5), hi=np.nanpercentile(boot[r], 97.5),
            phi=A.at[r, "phi"], ess_open=A.at[r, "ess_open"],
            ss=A.at[r, "ss"], elem=A.at[r, "elem"],
            measured=bool(pd.notna(A.at[r, "exp"])))
    for r in FOCUS}).T
BOOT["excludes_0"]  = (BOOT["lo"] > 0) | (BOOT["hi"] < 0)
BOOT["exceeds_glob"] = BOOT["lo"].astype(float) > A.attrs["dN_global"]
print("dN_global = %.1f\n" % A.attrs["dN_global"])
print(BOOT[["ss", "elem", "dN_HB", "lo", "hi", "phi", "ess_open", "measured",
            "excludes_0", "exceeds_glob"]].round(3).to_string())
print("\nphi > 1 with the interval clear of dN_global: %s"
      % (list(BOOT.index[BOOT["exceeds_glob"]]) or "none"))

## What this notebook finds

### The headline is a negative result

**No structural descriptor survives as an independent explanation.**
`hb_score_nat` is the only one significant under both analysis temperatures
($\rho$ = +0.66 and +0.45), and its partial correlation against $\Delta N_{HB}$
**collapses** once $\log p_{open}$ is held fixed: +0.24 (p = 0.19) at the
van't Hoff T and −0.04 (p = 0.82) at the hard-cut T. So it is not a cause; it is
a restatement of *amides that open often, open locally*. Since $\Delta N_{HB}$
and $\Delta G_{HX}$ already correlate at $\rho\approx0.88$, "which residues does
the model get wrong on cooperativity" and "which does it under-protect" are one
question, not two, and the descriptors tested here do not decompose it further.

**Explicitly not explanatory:** `sep` (H-bond partner sequence separation),
`nbr10` (burial), and `partner_frac` show no significant relation at either
temperature. The intuitive story — non-local H-bonds require global unfolding,
buried amides cannot fray — does **not** hold in ff_2.1.

### What the pattern does look like

- **Positional, not energetic.** Cooperativity rises with distance to the nearer
  end of the amide's own secondary-structure element: $\varphi$ = 0.23 at
  `elem_edge` 0 rising to 1.18 at 6. The same monotone trend appears at the
  hard-cut T (0.20 → 0.64) but `elem_edge` only reaches $p<0.05$ at the van't Hoff
  T, so it is a strong lead rather than an established descriptor.
- **Helix vs sheet.** H3 is 2–3× more cooperative than any strand
  ($\varphi \approx 0.61$ vs 0.20–0.30), and within it there is an N-to-C gradient:
  23–28 sit near $\varphi \approx 0.2$ while 29–31 reach 0.9–1.2. Fraying
  concentrates at the **helix N-terminal half**, the **β1 N-terminal edge** (3, 4),
  and the **β5 entry** (51, 53, 55) — the parent notebook's read, now with all 32
  natively H-bonded amides behind it instead of the 12 measured sites.
- **Two distinct "frayed" sets, and they are not the same list.** Raw
  $\Delta N_{HB} < 0$ (anti-cooperative — the open sub-ensemble is *more* H-bonded
  elsewhere than the closed one) gives 3, 4, 15, 53, 55, with bootstrap intervals
  excluding zero for 3, 4 and 15. Frayed *relative to structural context* gives
  4, 53, 27, 14, 28 — residue 3 drops out because its weak native H-bond score
  partly accounts for it, while helix residues 27 and 28 enter. The anti-cooperative
  set is where the parent's negative $m$-values come from.
- **The "over" side does not survive.** Only 29 and 30 reach $\varphi > 1$, and
  every bootstrap interval overlaps $\Delta N_{global}$ — *no residue is
  significantly more cooperative than global unfolding*. Their `ess_open` is 27 and
  31, so the values rest on few open configurations. "Upside over-predicts
  cooperativity for the helix core" is a lead, not a result.
- **The localisation is temperature-robust even though the fit is not.** The trend
  fit explains $R^2$ = 0.46 at the van't Hoff T but only 0.17 at the hard-cut T,
  yet the two temperatures rank the residuals at Spearman +0.73. Which residues are
  anomalous is stable; how much of $\Delta N_{HB}$ the descriptors capture is not.

### Q1, for the record

$f_{remain}$ = 0.961 over the whole ensemble against 0.795 restricted to the DSE.
Against DESMOND's 0.67 and experiment's $\approx 0$, that is two separate failures:
Upside's DSE carries *more* residual structure than the all-atom trajectories
Skinner criticised (same direction, worse), *and* under native conditions the
exchange flux runs through a local fray in an intact native state, which DESMOND
does not do.

## Caveats

- **The parent's prose and its code select different analysis temperatures**
  (0.7912 vs 0.8518, 44 K vs 23 K below $T_m$). Its step-5 diagnosis of the
  $m$-value failure — "~40 K below $T_m$", ESS$_{unfolded}\approx26$,
  $p_{unf}\approx6\times10^{-7}$ — is written for the hard-cut temperature. The
  lever-arm problem survives the change ($\Delta N_{closed}$ = 6.9 against the ~26
  needed), but those numbers need refreshing.
- $\Delta N_{HB}$ and $\varphi$ are free of the failed $s$ calibration by
  construction. `f_remain_dse` is not free of `UNFOLDED_CUT`, which defines the DSE.
- 6 descriptors × 2 temperatures, on 29–32 amides. Nothing is corrected for
  multiple comparisons beyond requiring both temperatures to agree, and the
  partial-correlation check is the only test here that distinguishes a cause from
  a correlate — it is the one that failed.
- `ess_open_dse` runs 10–197 at the van't Hoff T but only 2–17 at the hard-cut T,
  so the DSE-restricted comparison to DESMOND is usable only at the former.
- The bootstrap holds MBAR's $f_k$ fixed and blocks within rungs, so its intervals
  are a lower bound on the true uncertainty.
- Descriptors come from `COOP_STRIDE`-subsampled folded frames of replica 0 only.
  They are native-structure properties, so this is cheap rather than wrong, but a
  second replica would confirm it.

### The obvious next test

Every descriptor here is static — read off the native structure. The one that
would actually separate cause from correlate is **dynamic**: the lifetime and
cooperative extent of individual opening events, rather than their equilibrium
population. `dN_HB` averages over every open frame regardless of whether it belongs
to a 1-frame flicker or a 200-frame excursion, and Skinner's protection definition
explicitly carries a 0.5 ns memory for exactly that reason. Adding a dwell-time
threshold to `IS_OPEN` would test whether the local channel is a real structural
excursion or sub-resolution flickering that a memory criterion would discard.